In [55]:
import os
from dotenv import load_dotenv
from openai import OpenAI
from sklearn.metrics.pairwise import cosine_similarity, euclidean_distances, manhattan_distances
import numpy as np
import time

load_dotenv()

client = OpenAI(
    api_key=os.environ["OPENROUTER_API_KEY"],
    base_url="https://openrouter.ai/api/v1"
)

EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b:free"
print(EMBEDDING_MODEL)

nvidia/nemotron-3-embed-1b:free


In [56]:
documents = [
    {"id": 1,  "text": "Spring Boot simplifies building production-ready Java backend applications.",                                               "topic": "backend",           "level": "beginner"},
    {"id": 2,  "text": "Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.",                          "topic": "devops",            "level": "intermediate"},
    {"id": 3,  "text": "Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.",                                  "topic": "databases",         "level": "beginner"},
    {"id": 4,  "text": "Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.",                      "topic": "distributed-systems","level": "intermediate"},
    {"id": 5,  "text": "PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.",                  "topic": "databases",         "level": "beginner"},
    {"id": 6,  "text": "Vector databases store embeddings and make similarity search over high-dimensional vectors efficient.",                    "topic": "ai",                "level": "beginner"},
    {"id": 7,  "text": "RAG retrieves relevant external context and gives that context to a language model before generating an answer.",          "topic": "ai",                "level": "beginner"},
    {"id": 8,  "text": "An embedding model converts text into a dense numerical vector that captures semantic relationships.",                    "topic": "ai",                "level": "beginner"},
    {"id": 9,  "text": "Cosine similarity compares the angle between vectors and is commonly used for semantic text retrieval.",                  "topic": "ai",                "level": "intermediate"},
    {"id": 10, "text": "HNSW builds a navigable graph so approximate nearest-neighbor search can avoid scanning every vector.",                   "topic": "vector-search",     "level": "advanced"},
    {"id": 11, "text": "Exact nearest-neighbor search compares the query against every candidate vector to obtain the true top results.",         "topic": "vector-search",     "level": "intermediate"},
    {"id": 12, "text": "Quantization compresses vector representations so a vector database can reduce memory usage and improve search efficiency.","topic": "vector-search",     "level": "advanced"},
    {"id": 13, "text": "Payload indexes allow a vector database to combine semantic similarity with structured filters.",                         "topic": "vector-search",     "level": "advanced"},
    {"id": 14, "text": "Python is widely used for machine learning, data processing, and building AI applications.",                              "topic": "programming",       "level": "beginner"},
    {"id": 15, "text": "Async programming can improve throughput for applications that spend significant time waiting on network I/O.",           "topic": "programming",       "level": "intermediate"},
    {"id": 16, "text": "API gateways can provide routing, authentication, rate limiting, and observability for microservices.",                   "topic": "distributed-systems","level": "intermediate"},
    {"id": 17, "text": "Microservices split an application into independently deployable services that communicate over APIs or events.",         "topic": "distributed-systems","level": "intermediate"},
    {"id": 18, "text": "Observability combines logs, metrics, and traces to understand the behavior of distributed applications.",               "topic": "devops",            "level": "intermediate"},
    {"id": 19, "text": "A cache can reduce database load and latency when frequently requested data can be reused safely.",                       "topic": "databases",         "level": "beginner"},
    {"id": 20, "text": "Reranking can improve retrieval quality by applying a more expensive relevance model to an initial candidate set.",       "topic": "ai",                "level": "advanced"},
]

texts = [d["text"] for d in documents]
print(f"{len(documents)} documents")
print(texts[0])

20 documents
Spring Boot simplifies building production-ready Java backend applications.


In [57]:
def embed_documents(texts: list[str]) -> list[list[float]]:
    response = client.embeddings.create(model=EMBEDDING_MODEL, input=texts)
    return [d.embedding for d in response.data]


doc_embeddings = embed_documents(texts)

print("Number of embeddings:", len(doc_embeddings))
print("Dimensions per vector:", len(doc_embeddings[0]))
print("First 5 values:", doc_embeddings[0])


Number of embeddings: 20
Dimensions per vector: 2048
First 5 values: [0.09486015886068344, 0.07493623346090317, -0.0003346497833263129, -0.00322169647552073, 0.014210550114512444, -0.010696926154196262, 0.021084517240524292, -0.039743419736623764, 0.015385977923870087, -0.00482235848903656, 0.008787918835878372, -0.006052968092262745, 0.0022460301406681538, -0.007704064715653658, 0.010700485669076443, 0.0009034752729348838, 0.0008514582295902073, 0.005959317088127136, 0.03242503106594086, 0.009030005894601345, 0.0003809805493801832, 0.015569521114230156, -0.0179350133985281, 0.005743436980992556, 0.021483249962329865, -0.005791893694549799, -0.009824304841458797, -0.02709951251745224, 0.006302966270595789, 0.0038797222077846527, -0.020878031849861145, 0.00545131042599678, -0.0341833271086216, -0.014470438472926617, -0.0060340301133692265, -0.024358628317713737, -0.017685212194919586, 0.0020846386905759573, -0.01423527393490076, 0.0033797253854572773, 0.018665624782443047, 0.02187328040

In [58]:
query = "Which programming language is commonly used for AI applications?"

query_emb = embed_documents([query])

print("Query:", query)
print("Dimensions:", len(query_emb[0]))
print("Collection must use size =", len(doc_embeddings[0]))
print(query_emb)

Query: Which programming language is commonly used for AI applications?
Dimensions: 2048
Collection must use size = 2048
[[0.08161108940839767, 0.042288873344659805, 0.0155482804402709, 0.03575907647609711, -0.007101788185536861, -0.01083897240459919, 0.011618245393037796, 0.04856503754854202, 0.040427714586257935, 0.0005457531660795212, -0.010311586782336235, 0.0036016209051012993, 0.004340662155300379, 0.021298367530107498, -0.015651920810341835, -0.007206740789115429, -0.016565008088946342, -0.0024545772466808558, 0.01189986802637577, 0.00710878474637866, 0.03887966647744179, 0.021228399127721786, 0.02879197709262371, -0.01682126708328724, 0.017743974924087524, -0.030712608247995377, 0.013395441696047783, -0.0442742258310318, -0.004853617399930954, 0.012608298100531101, 0.026211895048618317, 0.0047683436423540115, 0.001983602764084935, -0.0011352365836501122, -0.03085254691541195, 0.01612420752644539, -0.004971251357346773, 0.0010810111416503787, 0.010730521753430367, -0.00745272310

In [59]:
# Metrics Calculations
cos_sim = cosine_similarity(doc_embeddings, query_emb)
dot_product = np.dot(np.array(doc_embeddings), np.array(query_emb).T)  # Transpose for dot product
l2_dist = euclidean_distances(doc_embeddings, query_emb)
l1_dist = manhattan_distances(doc_embeddings, query_emb)

# Print Header
print(f"{'ID':<4} {'Cosine Sim':<12} {'Dot Product':<12} {'L2 Dist':<10} {'L1 Dist':<10} Text")
print("-" * 110)

# Loop and Print Results for Each Document
for doc, c, d, l2, l1 in zip(documents, cos_sim, dot_product, l2_dist, l1_dist):
    # c, d, l2, l1 are arrays of size (1,), so [0] extracts the scalar float
    c_val, d_val, l2_val, l1_val = c[0], d[0], l2[0], l1[0]
    print(f"{doc['id']:<4} {c_val:<12.4f} {d_val:<12.4f} {l2_val:<10.4f} {l1_val:<10.4f} {doc['text']}")

ID   Cosine Sim   Dot Product  L2 Dist    L1 Dist    Text
--------------------------------------------------------------------------------------------------------------
1    0.1164       0.1164       1.3294     48.3115    Spring Boot simplifies building production-ready Java backend applications.
2    0.0775       0.0775       1.3583     48.7170    Kubernetes orchestrates containers and provides scheduling, service discovery, and self-healing.
3    0.1522       0.1522       1.3021     46.7003    Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.
4    0.1004       0.1004       1.3414     48.0236    Kafka is a distributed event streaming platform used to build high-throughput asynchronous systems.
5    0.0581       0.0581       1.3725     49.8218    PostgreSQL is a relational database with strong SQL support, transactions, indexing, and extensibility.
6    0.0615       0.0615       1.3701     49.7070    Vector databases store embeddings and make simil

In [60]:

def get_top_k(
    query: str,
    documents: list[dict],
    doc_embeddings: np.ndarray,
    k: int = 3,
    metric: str = "cosine"
) -> list[dict]:
    """
    Retrieves top K documents matching a query vector.
    
    Metrics available: 'cosine', 'dot', 'l2' (Euclidean), 'l1' (Manhattan)
    """
    # 1. Embed query and convert to numpy array
    query_emb = np.array(embed_documents([query]))

    # 2. Calculate distance/similarity scores
    if metric == "cosine":
        scores = cosine_similarity(doc_embeddings, query_emb).flatten()
        reverse_sort = True  # Higher score is better
    elif metric == "dot":
        scores = np.dot(doc_embeddings, query_emb.T).flatten()
        reverse_sort = True  # Higher score is better
    elif metric == "l2":
        scores = euclidean_distances(doc_embeddings, query_emb).flatten()
        reverse_sort = False # Lower distance is better
    elif metric == "l1":
        scores = manhattan_distances(doc_embeddings, query_emb).flatten()
        reverse_sort = False # Lower distance is better
    else:
        raise ValueError(f"Unsupported metric: {metric}")

    # 3. Pair documents with scores
    ranked_results = []
    for doc, score in zip(documents, scores):
        item = doc.copy()
        item["score"] = float(score)
        ranked_results.append(item)

    # 4. Sort and return top k
    ranked_results.sort(key=lambda x: x["score"], reverse=reverse_sort)
    return ranked_results[:k]

In [61]:
def retrieve(doc_embeds, query_embeds, docs, top_k=2):
    scores = cosine_similarity(doc_embeds, query_embeds).flatten()
    scored = list(zip(docs, scores))
    scored.sort(key=lambda x: x[1], reverse=True)
    return scored[:top_k]

top_k_res = retrieve(doc_embeddings, query_emb, documents)
for res in top_k_res:
    print(f"score:{res[1]}, doc: {res[0]} ")

score:0.6142459005131062, doc: {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'} 
score:0.19856391702761295, doc: {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'} 


## Qdrant Vector Database

In [62]:
from qdrant_client import QdrantClient, models

client = QdrantClient(url="http://localhost:6333")


In [63]:
COLLECTION="documents"
if client.collection_exists(COLLECTION):
    client.delete_collection(COLLECTION)

client.create_collection(
    collection_name=COLLECTION,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),
        distance=models.Distance.COSINE
    )
)

True

In [64]:
points = [
    models.PointStruct(
        id=i,
        vector=vec,
        payload={"text": doc, "category":"demo"}
    ) for i, (doc, vec) in enumerate(zip(documents, doc_embeddings))
]

client.upsert(collection_name=COLLECTION, points=points)

UpdateResult(operation_id=1, status=<UpdateStatus.COMPLETED: 'completed'>)

In [65]:
results = client.query_points(
    collection_name=COLLECTION,
    query=query_emb[0],
    limit=3,
    with_payload=True
).points

for p in results:
    print(p.score, p.payload)

0.6142462 {'text': {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}, 'category': 'demo'}
0.19856393 {'text': {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}, 'category': 'demo'}
0.15221941 {'text': {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}, 'category': 'demo'}


## Flat Search

In [66]:
flat_results = client.query_points(
    collection_name=COLLECTION,
    query=query_emb[0],
    limit=3,
    search_params=models.SearchParams(exact=True)
).points
for p in flat_results:
    print(p)

id=13 version=1 score=0.6142462 payload={'text': {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}, 'category': 'demo'} vector=None shard_key=None order_value=None
id=14 version=1 score=0.19856393 payload={'text': {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}, 'category': 'demo'} vector=None shard_key=None order_value=None
id=2 version=1 score=0.15221941 payload={'text': {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}, 'category': 'demo'} vector=None shard_key=None order_value=None


## ANN with HNSW

In [67]:
TUNED_DOC="tuned_documents"
if client.collection_exists(TUNED_DOC):
    client.delete_collection(TUNED_DOC)

client.create_collection(
    collection_name=TUNED_DOC,
    vectors_config=models.VectorParams(
        size=len(doc_embeddings[0]),
        distance=models.Distance.COSINE
    ),
    hnsw_config=models.HnswConfigDiff(m=16, ef_construct=100)
)

points = [
    models.PointStruct(
        id=i,
        vector=vec,
        payload={"text": doc, "category":"demo"}
    ) for i, (doc, vec) in enumerate(zip(documents, doc_embeddings))
]

client.upsert(collection_name=TUNED_DOC, points=points)

res = client.query_points(
    collection_name=TUNED_DOC,
    query=query_emb[0],
    limit=3,
    search_params=models.SearchParams(hnsw_ef=128)
).points

for p in res:
    print(p)

id=13 version=1 score=0.6142462 payload={'text': {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}, 'category': 'demo'} vector=None shard_key=None order_value=None
id=14 version=1 score=0.19856393 payload={'text': {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}, 'category': 'demo'} vector=None shard_key=None order_value=None
id=2 version=1 score=0.15221941 payload={'text': {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}, 'category': 'demo'} vector=None shard_key=None order_value=None


## Quantization

In [68]:
COMPRESSED_DOC = "compressed_documents"
if client.collection_exists(COMPRESSED_DOC):
    client.delete_collection(COMPRESSED_DOC)

client.create_collection(
    collection_name=COMPRESSED_DOC,
    vectors_config=models.VectorParams(size=len(doc_embeddings[0]), distance=models.Distance.COSINE),
    quantization_config=models.ScalarQuantization(
        scalar=models.ScalarQuantizationConfig(
            type=models.ScalarType.INT8,
            always_ram=True
        ))
)

points = [
    models.PointStruct(
        id=i,
        vector=vec,
        payload={"text": doc, "category":"demo"}
    ) for i, (doc, vec) in enumerate(zip(documents, doc_embeddings))
]
client.upsert(collection_name=COMPRESSED_DOC, points=points)


res = client.query_points(
    collection_name=COMPRESSED_DOC,
    query=query_emb[0],
    limit=3,
    with_payload=True
).points

for p in res:
    print(p)

id=13 version=1 score=0.6142462 payload={'text': {'id': 14, 'text': 'Python is widely used for machine learning, data processing, and building AI applications.', 'topic': 'programming', 'level': 'beginner'}, 'category': 'demo'} vector=None shard_key=None order_value=None
id=14 version=1 score=0.19856393 payload={'text': {'id': 15, 'text': 'Async programming can improve throughput for applications that spend significant time waiting on network I/O.', 'topic': 'programming', 'level': 'intermediate'}, 'category': 'demo'} vector=None shard_key=None order_value=None
id=2 version=1 score=0.15221941 payload={'text': {'id': 3, 'text': 'Redis is an in-memory data store commonly used for caching, sessions, and fast lookups.', 'topic': 'databases', 'level': 'beginner'}, 'category': 'demo'} vector=None shard_key=None order_value=None


## Metadata Filtering

In [69]:
client.create_payload_index(
    collection_name=COLLECTION,
    field_name="tenant_id",
    field_schema=models.PayloadSchemaType.KEYWORD
)

res = client.query_points(
    collection_name=COLLECTION,
    query=query_emb[0],
    limit=3,
    query_filter=models.Filter(
        must=[
            models.FieldCondition(key="tenant_id", match=models.MatchValue(value="company-123"))
        ]
    )
).points

for p in res:
    print(p)